# 🎬 AI-Powered Movie Recommendation System

**Beginner-friendly Content-Based Recommendation System**

This project uses the Kaggle **TMDB 5000 Movie Dataset** to recommend movies similar to a movie selected by the user.

### ML techniques used
- Data cleaning and preprocessing
- JSON metadata extraction
- Feature engineering
- NLP text vectorization with CountVectorizer
- Cosine similarity
- Content-based recommendation

### Project flow
`Raw movie data → Clean/parse metadata → Create tags → Vectorize → Cosine similarity → Top recommendations`

> Dataset source: Kaggle — TMDB 5000 Movie Dataset.


In [ ]:
# 1. Import libraries
import ast
import pickle
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity


## 2. Load the Kaggle datasets

Download `tmdb_5000_movies.csv` and `tmdb_5000_credits.csv` from Kaggle and place them in `data/`.

The dataset contains about 5,000 movies and includes movie metadata plus cast/crew information.


In [ ]:
movies = pd.read_csv("../data/tmdb_5000_movies.csv")
credits = pd.read_csv("../data/tmdb_5000_credits.csv")

print("Movies shape:", movies.shape)
print("Credits shape:", credits.shape)

movies.head()


In [ ]:
# 3. Merge the two datasets
# credits uses movie_id; movies uses id
credits = credits.rename(columns={"movie_id": "id"})

df = movies.merge(credits, on="id", how="inner")

print("Merged shape:", df.shape)
df[["id", "title_x", "overview"]].head()


In [ ]:
# 4. Rename title column and keep useful features
df = df.rename(columns={"title_x": "title"})

features = [
    "id", "title", "overview", "genres",
    "keywords", "cast", "crew",
    "vote_average", "vote_count", "popularity"
]

df = df[features].copy()

# Remove rows without essential recommendation information
df = df.dropna(subset=["title", "overview"])

print("Rows after cleaning:", len(df))
df.head()


## 5. Understand the JSON-like columns

Columns such as `genres`, `keywords`, `cast`, and `crew` contain lists stored as text.

Example:

`[{"id": 28, "name": "Action"}, {"id": 12, "name": "Adventure"}]`

We will extract only the useful names.


In [ ]:
def parse_names(text):
    """Extract the 'name' field from a JSON-like list stored as text."""
    try:
        items = ast.literal_eval(text)
        return [item["name"].replace(" ", "").lower()
                for item in items if "name" in item]
    except (ValueError, SyntaxError, TypeError):
        return []

def parse_cast(text, n=3):
    try:
        items = ast.literal_eval(text)
        return [item["name"].replace(" ", "").lower()
                for item in items[:n] if "name" in item]
    except (ValueError, SyntaxError, TypeError):
        return []

def parse_director(text):
    try:
        items = ast.literal_eval(text)
        for item in items:
            if item.get("job") == "Director":
                return [item["name"].replace(" ", "").lower()]
        return []
    except (ValueError, SyntaxError, TypeError):
        return []


In [ ]:
# 6. Create clean metadata features
df["genres"] = df["genres"].apply(parse_names)
df["keywords"] = df["keywords"].apply(parse_names)
df["cast"] = df["cast"].apply(parse_cast)
df["crew"] = df["crew"].apply(parse_director)

# Convert overview to individual lowercase words
df["overview"] = (
    df["overview"]
    .fillna("")
    .str.lower()
    .str.replace(r"[^a-zA-Z0-9 ]", " ", regex=True)
)

df.head()


## 7. Build the `tags` feature

The recommendation engine needs one combined text field describing each movie.

We combine:
- overview
- genres
- keywords
- top 3 cast members
- director

This lets the model compare movies using their content.


In [ ]:
df["tags"] = (
    df["overview"] + " " +
    df["genres"].apply(lambda x: " ".join(x)) + " " +
    df["keywords"].apply(lambda x: " ".join(x)) + " " +
    df["cast"].apply(lambda x: " ".join(x)) + " " +
    df["crew"].apply(lambda x: " ".join(x))
)

df["tags"] = df["tags"].str.replace(r"\s+", " ", regex=True).str.strip()

df[["title", "tags"]].head()


## 8. Convert movie text into numbers

Computers cannot directly compare sentences.

`CountVectorizer` converts words into numerical vectors.

We use at most 5,000 vocabulary terms and remove common English stop words.


In [ ]:
cv = CountVectorizer(max_features=5000, stop_words="english")

vectors = cv.fit_transform(df["tags"])

print("Vector matrix shape:", vectors.shape)


## 9. Calculate cosine similarity

Cosine similarity measures how close two movie vectors are.

- Close to `1` → very similar
- Close to `0` → not very similar

The result is a movie-by-movie similarity matrix.


In [ ]:
similarity = cosine_similarity(vectors)

print("Similarity matrix shape:", similarity.shape)


## 10. Recommendation function

The function:
1. Finds the selected movie's row.
2. Gets its similarity score against every movie.
3. Sorts movies from most similar to least similar.
4. Skips the selected movie itself.
5. Returns the top 5 recommendations.


In [ ]:
def recommend(movie, n=5):
    matches = df[df["title"].str.lower() == movie.lower()]

    if matches.empty:
        return [f"Movie '{movie}' was not found in the dataset."]

    movie_index = matches.index[0]
    distances = similarity[movie_index]

    movies_list = sorted(
        list(enumerate(distances)),
        reverse=True,
        key=lambda x: x[1]
    )[1:n+1]

    recommendations = []
    for index, score in movies_list:
        recommendations.append({
            "title": df.iloc[index]["title"],
            "similarity_score": round(float(score), 4)
        })

    return recommendations


In [ ]:
# 11. Test the recommender
recommend("Avatar")


## 12. Save model artifacts

The Streamlit app does not need to retrain the model every time. We save the processed movie table and similarity matrix.


In [ ]:
# Save artifacts one directory above this notebook's location
with open("../model/movies.pkl", "wb") as f:
    pickle.dump(df, f)

with open("../model/similarity.pkl", "wb") as f:
    pickle.dump(similarity, f)

with open("../model/vectorizer.pkl", "wb") as f:
    pickle.dump(cv, f)

print("Model artifacts saved.")


## 13. Simple model sanity check

A recommendation system is not a normal supervised ML problem, so there is no accuracy score like classification accuracy.

For this beginner project, we perform a sanity check:
- verify the requested movie exists
- verify recommendations are not the same movie
- inspect similarity scores


In [ ]:
test_movie = "Avatar"
results = recommend(test_movie, n=5)

print(f"Recommendations for: {test_movie}")
for item in results:
    print(f"- {item['title']} | similarity={item['similarity_score']}")
